# Lab 3 — Feedback Loops

**Goal:** Human corrections made at step 2 should automatically improve the agent's output at steps 3, 4, 5.  
This is the difference between a one-shot tool and a collaborative agent.

**Pattern:**
```
Step 1 → human approves
Step 2 → human approves WITH feedback: 'use simpler language'
Step 3 → agent automatically incorporates 'use simpler language'
Step 4 → same, no need to repeat yourself
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from hitl import Step, ApprovalGate, ApprovalResult, Decision, HITLAgent, FeedbackStore
from openai import OpenAI

client = OpenAI()
print('Ready ✓')

## 1. Show that feedback propagates

In [ ]:
feedback_store = FeedbackStore()

outputs = []
def executor_with_feedback(step: Step, feedback_ctx: str) -> str:
    msgs = [
        {'role': 'system', 'content': 'You are a content writer. Be concise (max 3 sentences).'},
    ]
    if feedback_ctx:
        msgs.append({'role': 'user', 'content': feedback_ctx})
    msgs.append({'role': 'user', 'content': step.description})

    resp = client.chat.completions.create(model='gpt-4o-mini', messages=msgs, max_tokens=200)
    output = resp.choices[0].message.content
    outputs.append({'step': step.name, 'feedback_ctx': feedback_ctx, 'output': output})
    return output


steps = [
    Step('intro',       'Write an intro about machine learning.',     confidence=0.9),
    Step('explanation', 'Explain how neural networks work.',          confidence=0.9),
    Step('examples',    'Give 3 real-world examples of ML.',          confidence=0.9),
    Step('conclusion',  'Write a conclusion for this ML article.',    confidence=0.9),
]

# Simulate: human gives feedback on step 2 to 'use simpler language for beginners'
call_count = [0]
def feedback_callback(step: Step) -> ApprovalResult:
    call_count[0] += 1
    if call_count[0] == 2:  # second step
        return ApprovalResult(Decision.APPROVE, feedback='Use simpler language — this is for beginners with no ML background.')
    return ApprovalResult(Decision.APPROVE)

gate = ApprovalGate(backend='callback', callback=feedback_callback, confidence_threshold=0.5)
agent = HITLAgent(gate=gate, feedback_store=feedback_store)
results = agent.run(steps, executor_with_feedback)

print('Feedback store contents:')
print(feedback_store.as_context())

## 2. Compare outputs before and after feedback

In [ ]:
for item in outputs:
    had_feedback = bool(item['feedback_ctx'])
    print(f"--- {item['step']} {'(with feedback context)' if had_feedback else '(no feedback)'} ---")
    print(item['output'][:300])
    print()

## 3. Persistent feedback across multiple corrections

In [ ]:
# Show that feedback accumulates over multiple human interactions
fs = FeedbackStore()
fs.add('intro',       'Original intro text',       'Make it warmer and more personal')
fs.add('explanation', 'Original explanation text',  'Add an analogy using cooking')
fs.add('examples',    'Original examples text',     'Focus on healthcare examples only')

print('Accumulated feedback context passed to subsequent steps:')
print(fs.as_context())

## Exercise
Build a **tone-locking** feedback system:  
If the human specifies a tone in any step (e.g. 'formal', 'casual', 'technical'), the agent should detect the tone preference and apply it to ALL future steps — even if the human doesn't mention it again.

**Hint:** After each human approval with feedback, use a small LLM call to extract tone preference if present.

In [ ]:
# Your code here
